In [36]:
import os 
import pandas as pd
import psycopg2 as pg 
from psycopg2 import extras #para poder insertar valores 
from dotenv import load_dotenv



In [37]:
load_dotenv()

conn = pg.connect(f"dbname={os.getenv('DB_NAME')} user={os.getenv('DB_USER')} password={os.getenv('DB_PASSWORD')} host={os.getenv('DB_HOST')} port={os.getenv('DB_PORT')}")
cur = conn.cursor()

In [38]:

path = "/home/Nahum/repos/idea_robust/etl/Setlist_completo.csv"

# CREATING THE TABLES
Aqui debo crear todas las tablas de golpe.

In [39]:
def sql_query(query:str ):
    cur.execute(query)
    return cur.fetchall()


In [40]:
def create_table(query:str, cur=cur, conn=conn, ):
    cur.execute(query)
    conn.commit()

In [41]:
schema = """
CREATE TABLE IF NOT EXISTS artist (
    id SERIAL PRIMARY KEY,
    name VARCHAR(60) UNIQUE NOT NULL
);

CREATE TABLE IF NOT EXISTS genre (
    id SERIAL PRIMARY KEY,
    name VARCHAR(100) UNIQUE NOT NULL
);

CREATE TABLE IF NOT EXISTS songs (
    id SERIAL PRIMARY KEY,
    title VARCHAR(60),
    artist_id INTEGER REFERENCES artist(id),
    genre_id INTEGER REFERENCES genre(id),
    tempo INTEGER CHECK (tempo > 0),
    tone VARCHAR(10),
    link_yt TEXT,
    UNIQUE (title, link_yt)
);

CREATE TABLE IF NOT EXISTS performance (
    id SERIAL PRIMARY KEY,
    song_id INTEGER REFERENCES songs(id),
    artist_id INTEGER REFERENCES artist(id),
    played_at DATE DEFAULT CURRENT_DATE,
    UNIQUE (played_at, song_id)
);
"""

# Creates all tables in a single query execution
create_table(schema)


# UPLOADIND DATA
## ARTISTS
debemos limpiar los nombres de los artistas y quitar los nulos

In [42]:

setlist = pd.read_csv(path)
setlist.keys()

Index(['Song', 'Artist', 'Genre', 'LastPlay', 'Tempo', 'TimesPlayed',
       'To-play', 'Tone', 'link'],
      dtype='str')

In [ ]:

setlist = setlist.drop(['To-play', 'LastPlay'], axis=1) #esta columna ya no la vamos a necesitar
setlist['Artist'] = setlist['Artist'].str.title() 
setlist['Artist'] = setlist['Artist'].str.strip() #Le hacemos un title para tener el mismo formato

In [44]:
artists = setlist['Artist'].unique().dropna() 
artists

<ArrowStringArray>
[          'Danilo Montero',             'Jaime Murrel',
              'Marcos Witt',     'Juan Carlos Alvarado',
                 'New Wine',              'Abel Zavala',
       'Church Of The City',           'Ingrid Rosario',
         'Marco Barrientos',  'En Espíritu Y En Verdad',
          'Miel San Marcos',               'Vino Nuevo',
          'Yashira Guidini',            'Generación 12',
                 'Manahaim',      'Jesús Adrián Romero',
 'Ericson Alexander Moreno',               'Montesanto',
            'Fenrel Monroy',            'Marcos Brunet',
          'Priscilla Bueno',            'Billy Bunster',
     'Alejandro Del Bosque',                    'Oasis',
             'Saraí Rivera',           'Coalo Zamorano',
          'Marcela Gándara',              'Alexis Peña',
                    'Ccint',         'Cristhian Hidall',
                    'Barak',                 'Ebenezer',
              'Avivamiento']
Length: 33, dtype: str

In [45]:
#psycopg only accepts list with tuples in
db_artist = [(artist, ) for artist in artists]
db_artist


[('Danilo Montero',),
 ('Jaime Murrel',),
 ('Marcos Witt',),
 ('Juan Carlos Alvarado',),
 ('New Wine',),
 ('Abel Zavala',),
 ('Church Of The City',),
 ('Ingrid Rosario',),
 ('Marco Barrientos',),
 ('En Espíritu Y En Verdad',),
 ('Miel San Marcos',),
 ('Vino Nuevo',),
 ('Yashira Guidini',),
 ('Generación 12',),
 ('Manahaim',),
 ('Jesús Adrián Romero',),
 ('Ericson Alexander Moreno',),
 ('Montesanto',),
 ('Fenrel Monroy',),
 ('Marcos Brunet',),
 ('Priscilla Bueno',),
 ('Billy Bunster',),
 ('Alejandro Del Bosque',),
 ('Oasis',),
 ('Saraí Rivera',),
 ('Coalo Zamorano',),
 ('Marcela Gándara',),
 ('Alexis Peña',),
 ('Ccint',),
 ('Cristhian Hidall',),
 ('Barak',),
 ('Ebenezer',),
 ('Avivamiento',)]

In [46]:

query = """
    INSERT INTO artist (name) 
    VALUES %s
    ON CONFLICT (name) DO NOTHING;
"""
extras.execute_values(cur, query, db_artist)
conn.commit()
    


In [47]:
db_artist = sql_query('SELECT * FROM artist LIMIT 4')
if db_artist:
    print('---------------- LA SUBIDA DE DATOS FUE EXITOSA')

---------------- LA SUBIDA DE DATOS FUE EXITOSA


## SUBIR LAS CANCIONES Y GENÉROS


In [48]:
genres = setlist['Genre'].value_counts()
genres

Genre
Alabanza     65
Adoración    38
Name: count, dtype: int64

In [49]:
genres = setlist['Genre'].unique().dropna()
genres_tuple = [(genre, ) for genre in genres] #psycopg mmangaes list of tuples

In [50]:
cur.executemany("""
    INSERT INTO genre (name)
    VALUES (%s)
    ON CONFLICT DO NOTHING;
""", genres_tuple)
conn.commit()

## filtrar datos con nan
se va a dividir en dos esta sección, ya que los nan tiene que ser removidos a manos, por el momento solo se subirán los que estén completos

In [51]:
rows_with_nan = setlist[setlist.isna().any(axis=1)]
print(f"hay {len(rows_with_nan)} filas con Nan que deben ser filtrados")
setlist_clean = setlist.dropna()
setlist_clean.info()

hay 31 filas con Nan que deben ser filtrados
<class 'pandas.DataFrame'>
Index: 72 entries, 0 to 101
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Song         72 non-null     str    
 1   Artist       72 non-null     str    
 2   Genre        72 non-null     str    
 3   LastPlay     72 non-null     str    
 4   Tempo        72 non-null     int64  
 5   TimesPlayed  72 non-null     float64
 6   Tone         72 non-null     str    
 7   link         72 non-null     str    
dtypes: float64(1), int64(1), str(6)
memory usage: 12.4 KB


songs_table = """
CREATE TABLE IF NOT EXISTS songs (
    id Serial PRIMARY KEY,
    title VARCHAR(60),
    artist_id INTEGER REFERENCES artist(id),
    genre_id INTEGER REFERENCES genre(id),
    tempo INTEGER CHECK (tempo>0),
    tone VARCHAR(10),
    link_yt TEXT
);

In [52]:

list_clean = list(setlist_clean[['Song', 'Artist', 'Genre', 'Tempo', 'Tone', 'link']].itertuples(index=False, name=None))
list_clean

[('Eres Todopoderoso',
  'Danilo Montero',
  'Alabanza',
  125,
  'G',
  'https://youtu.be/ZS7st5oNSWU?si=cBTdmBE8dB-LBaGS'),
 ('Oh Moradora de Sión',
  'Jaime Murrel',
  'Alabanza',
  135,
  'C-',
  'https://www.youtube.com/watch?v=9d0izkup40g'),
 ('Más El Dios de toda Gracia/Quiero Levantar mis manos',
  'Marcos Witt',
  'Adoración',
  62,
  'A',
  'https://youtu.be/tI70aR3lBIY?si=q0dgd4LqUBVi2Hpy'),
 ('Dios el Más grande',
  'Juan Carlos Alvarado',
  'Adoración',
  60,
  'Bb',
  'https://www.youtube.com/watch?v=MMGUo2xmmEY'),
 ('En Alta voz',
  'New Wine',
  'Alabanza',
  140,
  'Bb',
  'https://youtu.be/p4sEPZNHBb4?si=2zGm5ALvFtsCJzKd'),
 ('Enamórame ',
  'Abel Zavala',
  'Adoración',
  62,
  'G',
  'https://youtu.be/PcxOm1SMsNo?si=KNJBBk-ddKesGBZl'),
 ('Venció ',
  'Marcos Witt',
  'Alabanza',
  130,
  'C#-',
  'https://youtu.be/PH5lknwUIME?si=fc5FTx7ZSSYikszR'),
 ('La Bondad De Dios ',
  'Church Of The City',
  'Adoración',
  68,
  'G',
  'https://youtu.be/SnIzImY9wO4?si=saYxQYKU

To correct the table's references, we need to make subqueries to fetch the correct id

In [53]:


query = """
    INSERT INTO songs (title, artist_id, genre_id, tempo, tone, link_yt) 
    VALUES %s
    ON CONFLICT DO NOTHING;
"""

template = """(
    %s, 
    (SELECT (id) FROM artist WHERE name = %s LIMIT 1),
    (SELECT (id) FROM genre WHERE name = %s LIMIT 1),
    %s,
    %s,
    %s
)"""
#when usings subqueries, we must pass a template argument

try:
    extras.execute_values(cur, query, list_clean, template=template) #extra.execute_values inserts all the info at one time, not every row at a time.
    conn.commit()

except Exception as e:
    print(f"Detected Error: {e}")
    conn.rollback() #volver al estado anterior para volver a intentar la query

